# Module 03 — Storage I/O, Types & Iceberg (DSv2)

Compare formats and schemas, then a full Iceberg walkthrough: snapshots, time travel, partition evolution, and on-disk growth + compaction.

In [ ]:
import sys
sys.path.insert(0, '/workspace')
from shared.spark_session import begin_module, ui_links, explain_plan
from shared import config
from shared.utils import timed_action, timed_read, drop_iceberg_table, reset_spark_configs
from shared.demo_data import user_post_activity

spark = begin_module('module-03-storage-io')
ui_links()

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType
trips = spark.read.parquet(*config.yellow_taxi_paths())
sample = trips.select(
    'VendorID', 'PULocationID', 'total_amount', 'tpep_pickup_datetime'
).limit(200_000)
out = config.OUTPUT_BASE / '03-formats'
out.mkdir(parents=True, exist_ok=True)

## Write all formats

In [ ]:
sample.write.mode('overwrite').json(str(out / 'json'))
sample.write.mode('overwrite').option('header', True).csv(str(out / 'csv'))
sample.write.mode('overwrite').parquet(str(out / 'parquet'))
drop_iceberg_table(spark, 'local.default.io_lab_sample')
sample.writeTo('local.default.io_lab_sample').using('iceberg').create()

## Read benchmarks (count)

In [ ]:
results = {}
_, results['json'] = timed_read(spark, lambda: spark.read.json(str(out / 'json')))
_, results['csv'] = timed_read(
    spark, lambda: spark.read.option('header', True).schema(sample.schema).csv(str(out / 'csv'))
)
_, results['parquet'] = timed_read(spark, lambda: spark.read.parquet(str(out / 'parquet')))
_, results['iceberg'] = timed_read(spark, lambda: spark.table('local.default.io_lab_sample'))
for k, t in sorted(results.items(), key=lambda x: -x[1]):
    print(f'{k:10s} {t:.2f}s')

## Type safety — CSV without schema

In [ ]:
bad_csv = spark.read.option('header', True).csv(str(out / 'csv'))
bad_csv.printSchema()  # many strings
typed_csv = spark.read.schema(sample.schema).option('header', True).csv(str(out / 'csv'))
typed_csv.printSchema()

## Type safety — JSON inference

In [ ]:
infer_json = spark.read.json(str(out / 'json'))
infer_json.printSchema()
strict_json = spark.read.schema(sample.schema).json(str(out / 'json'))
strict_json.printSchema()

## Iceberg deep dive (single table)

Table `local.default.workshop_iceberg` — **DataSource V2**, snapshot isolation, hidden partitioning, metadata tree under `data/output/warehouse/`.

In [ ]:
ICEBERG = 'local.default.workshop_iceberg'
SHORT = 'workshop_iceberg'

def iceberg_storage_stats(short: str) -> dict:
    root = config.WAREHOUSE_DIR / 'default' / short
    if not root.exists():
        return {'data_files': 0, 'data_mb': 0.0, 'metadata_mb': 0.0, 'total_mb': 0.0}
    data = list((root / 'data').rglob('*.parquet')) if (root / 'data').exists() else []
    meta = list((root / 'metadata').rglob('*')) if (root / 'metadata').exists() else []
    data_b = sum(p.stat().st_size for p in data)
    meta_b = sum(p.stat().st_size for p in meta if p.is_file())
    return {
        'data_files': len(data),
        'data_mb': data_b / 1e6,
        'metadata_mb': meta_b / 1e6,
        'total_mb': (data_b + meta_b) / 1e6,
    }

def print_storage(label: str, stats: dict) -> None:
    print(
        f"{label}: {stats['data_files']} data files, "
        f"data {stats['data_mb']:.2f} MB, metadata {stats['metadata_mb']:.2f} MB, "
        f"total {stats['total_mb']:.2f} MB"
    )

def snapshot_ids(limit: int = 20):
    return spark.sql(
        f'SELECT snapshot_id, parent_id, committed_at FROM {ICEBERG}.snapshots '
        'ORDER BY committed_at'
    ).limit(limit).collect()

### Step 1 — initial load (partitioned table)

In [ ]:
drop_iceberg_table(spark, ICEBERG)
initial = sample.repartition(4).cache()
(
    initial.writeTo(ICEBERG)
    .using('iceberg')
    .partitionedBy('PULocationID')
    .create()
)
stats_initial = iceberg_storage_stats(SHORT)
print_storage('After initial create', stats_initial)
spark.sql(f'SELECT snapshot_id, summary FROM {ICEBERG}.snapshots').show(truncate=False)
snap_create = snapshot_ids()[-1].snapshot_id
rows_create = spark.table(ICEBERG).count()
print(f'snapshot {snap_create} row count: {rows_create:,}')

### Step 2 — mutate data (new snapshots)

Append rows, then **delete** — each commit adds a snapshot you can time-travel to.

In [ ]:
extra = sample.limit(25_000).repartition(8)
extra.writeTo(ICEBERG).using('iceberg').append()
snap_after_append = snapshot_ids()[-1].snapshot_id
print_storage('After append', iceberg_storage_stats(SHORT))

spark.sql(f'''
    DELETE FROM {ICEBERG}
    WHERE VendorID = 2 AND total_amount > 75
''')
snap_after_delete = snapshot_ids()[-1].snapshot_id
print_storage('After delete', iceberg_storage_stats(SHORT))
spark.sql(f'SELECT snapshot_id, committed_at FROM {ICEBERG}.snapshots ORDER BY committed_at').show()

### Step 3 — time travel

In [ ]:
def count_at_snapshot(snap_id: int) -> int:
    return spark.sql(f'SELECT COUNT(*) AS n FROM {ICEBERG} VERSION AS OF {snap_id}').collect()[0].n

for label, sid in [
    ('create', snap_create),
    ('after append', snap_after_append),
    ('after delete', snap_after_delete),
    ('current', None),
]:
    if sid is None:
        n = spark.table(ICEBERG).count()
    else:
        n = count_at_snapshot(sid)
    print(f'{label}: snapshot={sid} rows={n:,}')

# DataFrame reader variant
spark.read.option('snapshot-id', str(snap_create)).table(ICEBERG).count()

### Step 4 — partition evolution

Add a **new hidden partition field** — metadata only; old files keep the old spec, new writes use the new spec.

In [ ]:
spark.sql(f'DESCRIBE TABLE EXTENDED {ICEBERG}').filter(
    "col_name LIKE '%Partition%' or col_name = 'Provider'"
).show(truncate=False)
spark.sql(f'SELECT * FROM {ICEBERG}.partitions').show(10, truncate=False)

In [ ]:
spark.sql(f'ALTER TABLE {ICEBERG} ADD PARTITION FIELD bucket(16, VendorID)')
print('Added bucket(16, VendorID) — new appends land in evolved layout')
evo_batch = sample.limit(30_000).repartition(16, 'VendorID')
evo_batch.writeTo(ICEBERG).using('iceberg').append()
print_storage('After partition evolution + append', iceberg_storage_stats(SHORT))
spark.sql(f'SELECT partition, spec_id, record_count, file_count FROM {ICEBERG}.partitions').show(
    15, truncate=False
)

### Step 5 — repeated small writes (file sprawl)

Many tiny commits → more data files + metadata; compare volume before/after compaction.

In [ ]:
tiny = sample.limit(3_000)
for i in range(6):
    tiny.repartition(4).writeTo(ICEBERG).using('iceberg').append()
stats_before_compact = iceberg_storage_stats(SHORT)
print_storage('After repeated small appends', stats_before_compact)

In [ ]:
spark.sql('''
    CALL local.system.rewrite_data_files(
        table => 'default.workshop_iceberg',
        strategy => 'binpack',
        options => map(
            'rewrite-all', 'true',
            'min-input-files', '2'
        )
    )
''').show(truncate=False)

In [ ]:
stats_final = iceberg_storage_stats(SHORT)
print_storage('After rewrite_data_files', stats_final)
print('\nSummary')
for phase, st in [
    ('1 initial', stats_initial),
    ('2 after sprawl', stats_before_compact),
    ('3 after compaction', stats_final),
        ]:
    print_storage(phase, st)
wh = config.WAREHOUSE_DIR / 'default' / SHORT
print('warehouse path:', wh)
print('sample files:')
for p in sorted(wh.rglob('*'))[:25]:
    if p.is_file():
        print(f"  {p.relative_to(wh)}  ({p.stat().st_size / 1e3:.1f} KB)")

## Takeaways

1. Parquet/Iceberg win on analytics reads; enforce schemas on CSV/JSON.
2. **Snapshots** enable time travel (`VERSION AS OF` / `snapshot-id`).
3. **Partition evolution** changes layout for new files only.
4. Track **data vs metadata** footprint; compact with `rewrite_data_files` when file count explodes.